# Customer Churn Prediction - Feature Engineering

This notebook prepares the data for modeling:
- Drop redundant columns
- Encode categorical features
- Train/test split (stratified)
- Feature scaling
- Export processed data for the modeling notebook

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix TotalCharges (same as in EDA notebook)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

print(f"Shape: {df.shape}")
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Drop Redundant Columns

- **customerID**: unique identifier, no predictive value.
- **gender**: showed near-identical churn rates (~27%) for both groups in EDA - adds noise without signal.

In [4]:
# Drop columns with no predictive value
df = df.drop(columns=['customerID', 'gender'])

print(f"Shape after drop: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape after drop: (7043, 19)
Columns: ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [5]:
# Binary columns: Yes/No → 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df[binary_cols].head()

,Partner,Dependents,PhoneService,PaperlessBilling,Churn
0,1,0,0,1,0
1,0,0,1,0,0
2,0,0,1,1,1
3,0,0,0,0,0
4,0,0,1,1,1


In [6]:
# Categorical columns for One-Hot Encoding
categorical_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
                    'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                    'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

# One-Hot Encode (drop_first=True avoids redundancy)
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

print(f"Shape after encoding: {df.shape}")
df.head()

Shape after encoding: (7043, 30)


,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,0,1,0,1,29.85,29.85,0,1,...,0,0,0,0,0,0,0,0,1,0
1,0,0,0,34,1,0,56.95,1889.50,0,0,...,0,0,0,0,0,1,0,0,0,1
2,0,0,0,2,1,1,53.85,108.15,1,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,45,0,0,42.30,1840.75,0,1,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,2,1,1,70.70,151.65,1,0,...,0,0,0,0,0,0,0,0,1,0


## 2. Train/Test Split

We split the data into training (80%) and test (20%) sets. Two important choices:

- **Stratified split**: preserves the 73/27 class balance in both sets. Critical for imbalanced datasets - otherwise the test set could end up with very few positive examples.
- **random_state=42**: ensures reproducibility.

In [8]:
# Seperate feautures (X) and target (Y)

X = df.drop(columns=['Churn'])
y = df['Churn']

# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nChurn distribution in train: {y_train.mean():.4f}")
print(f"Churn distribution in test:  {y_test.mean():.4f}")

X_train: (5634, 29)
X_test:  (1409, 29)

Churn distribution in train: 0.2654
Churn distribution in test:  0.2654


## 3. Feature Scaling

We scale numerical features so they have mean=0, std=1. This matters for:
- **Logistic Regression**: features with larger ranges dominate the loss function.
- **Distance-based models** (KNN, SVM): require scaling.
- **Tree-based models** (Random Forest, XGBoost): don't need scaling, but it doesn't hurt.

**Critical: fit the scaler ONLY on training data, then transform both train and test.**
Fitting on the full dataset would leak information from test into training.

In [9]:
# Numerical features to scale (others are already 0/1)
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler()

# Fit on train only, transform both
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

X_train[numerical_cols].describe().round(3)

,tenure,MonthlyCharges,TotalCharges
count,5634.000,5634.000,5634.000
mean,-0.000,-0.000,0.000
std,1.000,1.000,1.000
min,-1.322,-1.544,-1.009
25%,-0.956,-0.971,-0.832
50%,-0.142,0.185,-0.397
75%,0.916,0.832,0.674
max,1.608,1.786,2.802


## 4. Export Processed Data

Save the prepared train/test sets to disk so the modeling notebook can load them directly.

In [10]:
import os

# Create processed data folder if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save train/test sets
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Saved:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

Saved:
  X_train: (5634, 29)
  X_test:  (1409, 29)
  y_train: (5634,)
  y_test:  (1409,)
